# Практическая работа. Кросс-валидация и подбор гиперпараметров
## «Один экзамен – случайность, пять экзаменов – оценка»

**Цель:** выяснить, почему оценка качества по одной валидационной выборке ненадежна, написать на чистом Python K-блочную и стратифицированную кросс-валидацию и скользящий контроль по одному объекту, подобрать по сетке число соседей $k$ для kNN (Б1.2) и глубину дерева решений (Б1.4), встроить подготовку данных из Б1.6 внутрь каждого блока и своими руками увидеть утечку, которая дает 80 % точности на случайных числах.

> **Правило работы.** Внутри своих функций – только списки, словари, циклы, `math` и `random`. Никаких `numpy`, `pandas` и `sklearn`: библиотека `sklearn` нужна только для загрузки набора Wine и для сверки в последнем задании, `matplotlib` – только в готовых ячейках с графиками.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. После функций идут строки проверки: напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию ниже – все задания опираются на нее.

**План работы**

| Задание | Что пишем | Главный результат |
|---|---|---|
| К1 | `mean_std`, `holdout_score` | одна валидационная выборка дает от 79 % до 95 % |
| К2 | `k_fold_indices`, `split_by_fold` | номера объектов нарезаны на $K$ блоков |
| К3 | `cross_val_score` | оценка kNN кросс-валидацией: 87,6 % ± 2,3 % |
| К4 | `stratified_k_fold_indices`, `count_in_folds` | редкий класс есть в каждом блоке |
| К5 | `loo_folds`, сравнение разных $K$ | сколько блоков взять |
| К6 | `make_grid`, `grid_search` | лучшее $k$ для kNN |
| К7 | перебор глубины дерева на Wine | лучшая глубина и итоговая проверка на тесте |
| К8 | `cross_val_score_prep` | подготовка данных из Б1.6 внутри блоков |
| К9 | `fit_selector`, `apply_selector` | утечка через отбор признаков |
| К10 | сто случайных разбиений | разброс оценки в 4,4 раза меньше |
| К11 | полный протокол и сверка со sklearn | итоговая модель и честная оценка на тесте |

## Теория: как честно оценить модель и выбрать ее настройки

**1. Параметры и гиперпараметры.** **Параметры** модель находит сама во время обучения: веса логистической регрессии (Б1.3), вопросы и пороги в узлах дерева решений (Б1.4). **Гиперпараметры** задает человек до обучения: число соседей $k$ и вид расстояния в kNN, максимальную глубину дерева `max_depth`, скорость обучения градиентного спуска. Выбрать гиперпараметры по обучающей выборке нельзя: на ней kNN с $k = 1$ и дерево без ограничения глубины всегда получают 100 %, хотя на новых данных ошибаются чаще других. Поэтому гиперпараметры выбирают по качеству на объектах, которых модель при обучении не видела.

**2. Беда одной валидационной выборки.** В Б1.2 мы отрезали от обучающей выборки 63 объекта (валидационную выборку) и выбрали $k = 9$. Но какие объекты попали в валидацию, решил случай: число `seed` в перемешивании. Один объект из 63 – это $1/63 \approx 1{,}6\,\%$ точности, поэтому два-три трудных объекта, случайно попавших в валидацию, заметно сдвигают оценку. Говорят, что у такой оценки большой **разброс**: повторите разбиение с другим `seed` – получите другое число и, возможно, другое лучшее $k$.

**3. K-блочная кросс-валидация** (перекрестная проверка, по-английски K-fold cross-validation). Число блоков обозначаем большой буквой $K$, чтобы не путать с числом соседей $k$.

1. Перемешать номера объектов обучающей выборки и разрезать их на $K$ блоков почти равного размера.
2. Сделать $K$ раундов. В раунде $i$ блок $i$ служит валидационным, а остальные $K - 1$ блоков – обучающими. Обучить модель и посчитать точность $a_i$ на блоке $i$.
3. Итоговая оценка – среднее, а разброс по блокам показывает стандартное отклонение:

$$\mathrm{CV} = \frac{1}{K}\sum_{i=1}^{K} a_i, \qquad \sigma = \sqrt{\frac{1}{K}\sum_{i=1}^{K} (a_i - \mathrm{CV})^2}.$$

Каждый объект ровно один раз побывал в валидации и $K - 1$ раз в обучении. Оценка опирается на все 210 объектов, а не на 63, поэтому гораздо меньше зависит от случая.

*Размеры блоков.* Если $n$ не делится на $K$ нацело, первые $n \bmod K$ блоков получают на один объект больше: размер блока $i$ равен $\lfloor n / K \rfloor + 1$ при $i < n \bmod K$ и $\lfloor n / K \rfloor$ иначе. Пример: $n = 11$, $K = 3$ дает блоки из 4, 4 и 3 объектов. Так же режет данные `KFold` из sklearn.

**4. Стратифицированная кросс-валидация.** Если какой-то класс редкий, при обычной нарезке он может целиком не попасть в один из блоков: в этом раунде модель вообще не проверяется на этом классе. Стратифицированная нарезка сохраняет доли классов в каждом блоке. Алгоритм: разложить номера объектов по классам, перемешать номера внутри каждого класса и раздавать их по блокам по кругу, как карты игрокам. Для классификации это способ по умолчанию; в Б1.1 мы так же стратифицировали разделение на обучающую и тестовую выборки.

**5. Скользящий контроль по одному объекту** (по-английски leave-one-out). Это кросс-валидация с $K = n$: в каждом раунде валидация – один объект, обучение – все остальные. Случайности нет (перемешивать нечего), обучение идет почти на всех данных. Цена – $n$ обучений: для 210 объектов модель обучается 210 раз. Для kNN это недорого, потому что обучения у него нет, а для медленных моделей слишком долго. На практике обычно берут $K = 5$ или $K = 10$.

**6. Перебор по сетке.** Для каждого гиперпараметра задаем список значений, а **сетка** – это все их комбинации. Для каждой комбинации считаем CV и выбираем комбинацию с наибольшим CV. Число обучений:

$$\text{число обучений} = \text{число комбинаций} \times K.$$

Пример: 20 значений $k$, 2 вида расстояния и 5 блоков дают $20 \cdot 2 \cdot 5 = 200$ обучений. Сетка растет как произведение: добавили третий гиперпараметр с 10 значениями – обучений стало в 10 раз больше.

**7. Подготовка данных внутри каждого блока.** Все, что вычисляется по данным (минимум и максимум, медиана, среднее $\mu$ и стандартное отклонение $\sigma$, границы выбросов, выбор полезных признаков), тоже является обучением. В кросс-валидации роль теста играет валидационный блок, поэтому параметры подготовки считаем только по обучающей части раунда и применяем к валидационному блоку. Иначе возникает **утечка**: сведения из валидационного блока попадают в модель, и оценка получается завышенной.

**8. Полный протокол.**

1. Отложить тестовую выборку и не трогать ее до конца.
2. На обучающей выборке перебрать гиперпараметры по сетке с кросс-валидацией.
3. Обучить модель с лучшими гиперпараметрами на **всей** обучающей выборке.
4. Один раз проверить модель на тестовой выборке: это и есть оценка качества на новых данных.

Лучшее значение CV по сетке немного завышено: мы выбрали максимум из многих чисел, каждое из которых колеблется случайно. Честную оценку всей процедуры выбора дает тестовая выборка или вложенная кросс-валидация (задание повышенной сложности).

**Где почитать.** Я. Гудфеллоу, И. Бенджио, А. Курвилль «Глубокое обучение»: раздел 5.3 «Гиперпараметры и контрольные наборы» (валидационную выборку там называют контрольным набором) и раздел 5.3.1 «Перекрестная проверка» с алгоритмом 5.1 – это ровно та функция `cross_val_score`, которую мы напишем в задании К3.

In [ ]:
# Готовая ячейка: библиотеки, данные и разделение из прошлых работ
import math
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine     # только для загрузки набора Wine

RANDOM_STATE = 42
BLUE, NAVY, GREY, DARK_BLUE = '#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9'   # цвета графиков

def train_test_split(X, y, test_size=0.3, seed=RANDOM_STATE):
    """Перемешивает объекты и делит их на обучающую и тестовую части (из Б1.2)."""
    idx = list(range(len(X)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(X) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

def make_road_objects(n_per_class=100, seed=RANDOM_STATE):
    """Объекты, которые «видит» беспилотник (из Б1.2): 0 – пешеход, 1 – велосипедист, 2 – автомобиль.
    Признаки: [скорость, км/ч;  длина, м;  высота, м;  расстояние до объекта, м]."""
    rnd = random.Random(seed)
    params = {0: ((5, 2.5), (0.7, 0.4), (1.7, 0.15)),
              1: ((14, 6), (1.7, 0.45), (1.75, 0.15)),
              2: ((18, 12), (3.6, 1.1), (1.6, 0.2))}
    X, y = [], []
    for c, (pv, pl, ph) in params.items():
        for _ in range(n_per_class):
            v = max(0.5, rnd.gauss(*pv))
            L = max(0.2, rnd.gauss(*pl))
            H = max(0.8, rnd.gauss(*ph))
            D = rnd.uniform(5, 80)
            X.append([round(v, 1), round(L, 2), round(H, 2), round(D, 1)])
            y.append(c)
    idx = list(range(len(X)))
    rnd.shuffle(idx)
    return [X[i] for i in idx], [y[i] for i in idx]

def minmax_fit(X):
    """Минимум и максимум каждого признака (из Б1.2)."""
    mins = [min(row[j] for row in X) for j in range(len(X[0]))]
    maxs = [max(row[j] for row in X) for j in range(len(X[0]))]
    return mins, maxs

def minmax_transform(X, mins, maxs):
    return [[(v - lo) / (hi - lo) for v, lo, hi in zip(row, mins, maxs)] for row in X]

CLASS_NAMES = ['пешеход', 'велосипедист', 'автомобиль']

# Данные беспилотника, как в Б1.2: 210 объектов в обучающей выборке, 90 в тестовой.
X_road, y_road = make_road_objects()
X_train, X_test, y_train, y_test = train_test_split(X_road, y_road)
# Оставляем три полезных признака (скорость, длина, высота) и нормализуем их, как в конце Б1.2.
# Здесь min и max посчитаны по всей обучающей выборке; почему правильнее считать их
# внутри каждого блока кросс-валидации, разберем в задании К8.
X_train3, X_test3 = [row[:3] for row in X_train], [row[:3] for row in X_test]
mins, maxs = minmax_fit(X_train3)
Xr_train, Xr_test = minmax_transform(X_train3, mins, maxs), minmax_transform(X_test3, mins, maxs)

# Набор Wine, как в Б1.6: 125 вин в обучающей выборке, 53 в тестовой, три сорта, 13 признаков.
wine = load_wine()
X_wine, y_wine = wine.data.tolist(), wine.target.tolist()
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_wine, y_wine)

print('Беспилотник:', len(Xr_train), 'и', len(Xr_test), '  ожидается 210 и 90')
print('Wine:       ', len(Xw_train), 'и', len(Xw_test), '  ожидается 125 и 53')

**Модель как функция.** Чтобы одна функция кросс-валидации работала с любой моделью, договоримся: модель – это функция

```python
model(X_train, y_train, X_val, params)   # обучается на X_train, y_train и возвращает прогнозы для X_val
```

Гиперпараметры передаются словарем `params`, например `{'k': 9}` для kNN или `{'max_depth': 3}` для дерева. Функцию можно передать другой функции как аргумент – так же, как в Б1.2 мы передавали `dist=euclidean` в `get_neighbors`.

В готовой ячейке ниже – kNN из Б1.2 и дерево решений из Б1.4 (вы их уже писали) и две «обертки» по этому договору: `knn_model` и `tree_model`.

In [ ]:
# Готовая ячейка: модели из прошлых работ
# ---- kNN (Б1.2)
def euclidean(a, b):
    return sum((ai - bi) ** 2 for ai, bi in zip(a, b)) ** 0.5

def manhattan(a, b):
    return sum(abs(ai - bi) for ai, bi in zip(a, b))

def get_neighbors(X_train, y_train, x, k, dist=euclidean):
    pairs = sorted(((dist(xi, x), yi) for xi, yi in zip(X_train, y_train)), key=lambda p: p[0])
    return [yi for d, yi in pairs[:k]]

def majority_vote(labels):
    counts = {}
    for lab in labels:
        counts[lab] = counts.get(lab, 0) + 1
    best = max(counts.values())
    for lab in labels:                      # при ничьей побеждает ближний сосед
        if counts[lab] == best:
            return lab

def knn_predict(X_train, y_train, X_test, k, dist=euclidean):
    return [majority_vote(get_neighbors(X_train, y_train, x, k, dist)) for x in X_test]

def accuracy(y_true, y_pred):
    return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

# ---- дерево решений (Б1.4), критерий Джини
def count_classes(y):
    counts = {}
    for label in y:
        counts[label] = counts.get(label, 0) + 1
    return counts

def gini(y):
    n = len(y)
    return 0.0 if n == 0 else 1 - sum((c / n) ** 2 for c in count_classes(y).values())

def split(X, y, feature, threshold):
    X_left, y_left, X_right, y_right = [], [], [], []
    for x, label in zip(X, y):
        if x[feature] <= threshold:
            X_left.append(x); y_left.append(label)
        else:
            X_right.append(x); y_right.append(label)
    return X_left, y_left, X_right, y_right

def best_split(X, y):
    best_feature, best_threshold, best_gain = None, None, 0.0
    for feature in range(len(X[0])):
        values = sorted(set(x[feature] for x in X))
        for t in [(values[i] + values[i + 1]) / 2 for i in range(len(values) - 1)]:
            _, y_left, _, y_right = split(X, y, feature, t)
            n = len(y)
            gain = gini(y) - (len(y_left) / n * gini(y_left) + len(y_right) / n * gini(y_right))
            if gain > best_gain:
                best_feature, best_threshold, best_gain = feature, t, gain
    return best_feature, best_threshold, best_gain

def build_tree(X, y, depth=0, max_depth=3):
    counts = count_classes(y)
    leaf = {'leaf': True, 'class': max(counts, key=counts.get), 'n': len(y)}
    if depth >= max_depth or len(y) < 2 or len(counts) == 1:
        return leaf
    feature, threshold, gain = best_split(X, y)
    if feature is None:
        return leaf
    X_left, y_left, X_right, y_right = split(X, y, feature, threshold)
    return {'leaf': False, 'feature': feature, 'threshold': threshold,
            'left': build_tree(X_left, y_left, depth + 1, max_depth),
            'right': build_tree(X_right, y_right, depth + 1, max_depth)}

def predict(tree, X):
    preds = []
    for x in X:
        node = tree
        while not node['leaf']:
            node = node['left'] if x[node['feature']] <= node['threshold'] else node['right']
        preds.append(node['class'])
    return preds

# ---- обертки по договору model(X_train, y_train, X_val, params)
DISTANCES = {'euclidean': euclidean, 'manhattan': manhattan}

def knn_model(X_train, y_train, X_val, params):
    """kNN: params['k'] – число соседей, params['dist'] – название расстояния (по умолчанию евклидово)."""
    return knn_predict(X_train, y_train, X_val, params['k'], DISTANCES[params.get('dist', 'euclidean')])

def tree_model(X_train, y_train, X_val, params):
    """Дерево решений: params['max_depth'] – максимальная глубина."""
    return predict(build_tree(X_train, y_train, max_depth=params['max_depth']), X_val)

print(knn_model(Xr_train, y_train, Xr_test[:5], {'k': 9}), '  ожидается [0, 0, 1, 0, 0]')
print(tree_model(Xw_train, yw_train, Xw_test[:5], {'max_depth': 3}), '  ожидается [2, 0, 0, 1, 0]')

**Задание К1. Разброс одной валидационной выборки.**

Сначала повторим эксперимент из Б1.2 много раз и посмотрим, насколько «прыгает» оценка.

- `mean_std(values)` – возвращает пару `(среднее, стандартное отклонение)` списка чисел по формулам из п. 3 теории (делим на $n$, как в Б1.6). Корень – `math.sqrt`.
- `holdout_score(model, X, y, params, seed, val_size=0.3)` – оценка на одной валидационной выборке: разделить `X, y` функцией `train_test_split` с данным `seed` и долей `val_size`, обучить модель на обучающей части, сделать прогноз для валидационной и вернуть точность.

Затем посчитайте `holdout_score` для kNN с $k = 9$ при `seed` от 0 до 99 и сложите 100 оценок в список `hold_scores`.

*Пример для проверки на бумаге.* Для чисел $2, 4, 4, 4, 5, 5, 7, 9$ среднее $40 / 8 = 5$, сумма квадратов отклонений $9 + 1 + 1 + 1 + 0 + 0 + 4 + 16 = 32$, $\sigma = \sqrt{32 / 8} = 2$.

In [ ]:
def mean_std(values):
    m = ...                                     # <-- среднее
    s = ...                                     # <-- стандартное отклонение
    return m, s

def holdout_score(model, X, y, params, seed, val_size=0.3):
    X_tr, X_val, y_tr, y_val = ...              # <-- разделение с данным seed
    y_pred = ...                                # <-- обучение и прогноз для валидации
    return ...                                  # <-- точность на валидации


print(mean_std([2, 4, 4, 4, 5, 5, 7, 9]), '  ожидается (5.0, 2.0)')
print(round(holdout_score(knn_model, Xr_train, y_train, {'k': 9}, seed=13), 3), '  ожидается 0.921 – это валидация из Б1.2')

hold_scores = []
for seed in range(100):
    ...                                         # <-- оценка при данном seed

m, s = mean_std(hold_scores)
print(f'100 разбиений: от {min(hold_scores):.3f} до {max(hold_scores):.3f}, среднее {m:.3f}, σ = {s:.3f}')
print('  ожидается: от 0.794 до 0.952, среднее 0.869, σ = 0.036')

In [ ]:
# Готовая ячейка: гистограмма ста оценок
plt.figure(figsize=(7, 3.3))
bins = [(j - 0.5) / 63 for j in range(49, 62)]         # одна «ступенька» = один объект из 63
plt.hist(hold_scores, bins=bins, color=BLUE, edgecolor='white')
plt.axvline(0.921, color=NAVY, linestyle='--', label='наша валидация из Б1.2 (seed = 13)')
plt.xlabel('точность kNN (k = 9) на валидационной выборке из 63 объектов')
plt.ylabel('сколько разбиений')
plt.title('Одна модель, сто случайных разбиений')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Готовая ячейка: какое k выберет одна валидационная выборка при разных seed (функция choose_k из Б1.2)
def choose_k(X_tr, y_tr, X_val, y_val, ks):
    errors = [1 - accuracy(y_val, knn_predict(X_tr, y_tr, X_val, k)) for k in ks]
    return ks[errors.index(min(errors))]

ks = list(range(1, 40, 2))
chosen = []
for seed in range(20):
    X_tr, X_val, y_tr, y_val = train_test_split(Xr_train, y_train, test_size=0.3, seed=seed)
    chosen.append(choose_k(X_tr, y_tr, X_val, y_val, ks))
print('лучшее k при seed = 0, 1, ..., 19:', chosen)
print('  ожидается [9, 3, 11, 11, 5, 9, 15, 9, 19, 13, 9, 9, 3, 9, 11, 9, 5, 19, 17, 3]')

**Вопрос К1.** а) Модель и данные одни и те же. Почему точность меняется от 79 % до 95 %? б) Какое $k$ «правильное», если при разных `seed` лучшими оказываются и 3, и 19? в) Какой вывод о числе 0,921, которое мы получили в Б1.2?

*Ваш ответ:*

**Задание К2. Нарезаем блоки.**

- `k_fold_indices(n, n_folds, seed)` – перемешать номера `0 … n-1` через `random.Random(seed).shuffle(idx)` и разрезать их на `n_folds` блоков по правилу из п. 3 теории: блок `i` имеет размер `n // n_folds`, и еще один объект добавляется, если `i < n % n_folds`. Удобно хранить переменную `start` – с какого места списка `idx` начинается очередной блок. Возвращает список блоков, каждый блок – список номеров.
- `split_by_fold(X, y, folds, i)` – раунд `i`: валидационные номера – `folds[i]`, обучающие – номера из всех остальных блоков. Возвращает `X_tr, y_tr, X_val, y_val`.

*Пример.* $n = 10$, $K = 3$: размеры блоков $4, 3, 3$ (одному блоку достался остаток $10 \bmod 3 = 1$). Если после перемешивания `idx = [7, 8, 1, 5, 3, 4, 2, 0, 9, 6]`, то блоки: `[7, 8, 1, 5]`, `[3, 4, 2]`, `[0, 9, 6]`.

In [ ]:
def k_fold_indices(n, n_folds, seed=RANDOM_STATE):
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    folds, start = [], 0
    for i in range(n_folds):
        size = ...                              # <-- размер блока i
        ...                                     # <-- срез списка idx
        start += size
    return folds

def split_by_fold(X, y, folds, i):
    val_idx = folds[i]
    train_idx = []
    for f in range(len(folds)):
        if f != i:
            train_idx += ...               # <-- номера из всех блоков, кроме i-го
    X_tr = ...                             # <-- объекты обучающей части
    y_tr = ...                             # <-- ответы обучающей части
    X_val = ...                            # <-- объекты валидационного блока
    y_val = ...                            # <-- ответы валидационного блока
    return X_tr, y_tr, X_val, y_val


print(k_fold_indices(10, 3, seed=0), '  ожидается [[7, 8, 1, 5], [3, 4, 2], [0, 9, 6]]')
print([len(f) for f in k_fold_indices(11, 3)], '  ожидается [4, 4, 3]')

folds5 = k_fold_indices(len(Xr_train), 5)
print([len(f) for f in folds5], '  ожидается [42, 42, 42, 42, 42]')
print(sorted(j for f in folds5 for j in f) == list(range(210)), '  ожидается True: каждый объект ровно в одном блоке')
X_tr, y_tr, X_val, y_val = split_by_fold(Xr_train, y_train, folds5, 0)
print(len(X_tr), len(X_val), '  ожидается 168 42')

In [ ]:
# Готовая ячейка: схема пяти раундов (синий – валидационный блок, серый – обучающие блоки)
fig, ax = plt.subplots(figsize=(7, 2.6))
for r in range(5):
    for b in range(5):
        ax.add_patch(plt.Rectangle((b * 42, 4 - r), 40, 0.8, color=BLUE if b == r else '#E1E3E5'))
    ax.text(-8, 4 - r + 0.4, f'раунд {r + 1}', ha='right', va='center', fontsize=9)
ax.set_xlim(-60, 212); ax.set_ylim(-0.2, 5)
ax.axis('off'); ax.set_title('5 блоков по 42 объекта: каждый блок один раз служит валидацией')
plt.show()

**Вопрос К2.** а) Зачем перемешивать номера перед нарезкой? Что будет, если данные записаны по порядку классов (сначала все пешеходы, потом велосипедисты, потом автомобили), а мы нарежем их без перемешивания на 3 блока? б) Сколько раз каждый объект побывает в обучающей части за 5 раундов?

*Ваш ответ:*

**Задание К3. Кросс-валидация.**

`cross_val_score(model, X, y, folds, params)` – для каждого раунда `i` получить обучающую часть и валидационный блок (`split_by_fold`), обучить модель и сделать прогноз для блока, посчитать точность и сложить ее в список `scores`. Функция возвращает список из $K$ точностей – по одной на раунд. Среднее и разброс посчитает `mean_std` из К1.

Модель передается как аргумент `model` – внутри функции вызывайте ее как `model(X_tr, y_tr, X_val, params)`. Поэтому одна и та же `cross_val_score` подойдет и для kNN, и для дерева.

In [ ]:
def cross_val_score(model, X, y, folds, params):
    scores = []
    for i in range(len(folds)):
        X_tr, y_tr, X_val, y_val = ...          # <-- раунд i: блок i – валидация
        y_pred = ...                            # <-- обучение на остальных блоках, прогноз для блока i
        ...                                     # <-- точность на блоке i
    return scores


scores = cross_val_score(knn_model, Xr_train, y_train, folds5, {'k': 9})
m, s = mean_std(scores)
print('точность по блокам:', [round(a, 3) for a in scores])
print('  ожидается [0.881, 0.905, 0.881, 0.833, 0.881]')
print(f'CV = {m:.3f} ± {s:.3f}   ожидается 0.876 ± 0.023')
print('для сравнения: одна валидация из Б1.2 дала 0.921')

**Вопрос К3.** а) Почему оценка кросс-валидацией (0,876) ниже, чем одна валидация из Б1.2 (0,921)? Какой из двух оценок вы больше доверяете и почему? б) Что показывает $\sigma = 0{,}023$: разброс качества модели или разброс сложности блоков? в) Сколько раз обучалась модель, чтобы получить одно число CV?

*Ваш ответ:*

**Задание К4. Редкий класс и стратификация.**

Представим, что велосипедистов на дороге мало: в готовой ячейке ниже из обучающей выборки оставлены только первые 10 велосипедистов из 69. Получилась выборка `X_rare, y_rare` из 151 объекта: 67 пешеходов, 10 велосипедистов, 74 автомобиля.

- `count_in_folds(y, folds, label)` – список: сколько объектов класса `label` попало в каждый блок.
- `stratified_k_fold_indices(y, n_folds, seed)` – стратифицированная нарезка (п. 4 теории):
  1. разложить номера объектов по классам в словарь `by_class` (ключ – класс, значение – список номеров); выражение `by_class.setdefault(label, [])` вернет список класса `label`, а при первой встрече сначала создаст пустой;
  2. для каждого класса по порядку (`sorted(by_class)`) перемешать его номера генератором `rnd` и раздавать их по блокам по кругу: номер попадает в блок `pos % n_folds`, где `pos` – сквозной счетчик розданных номеров.

Сквозной счетчик нужен, чтобы блоки получились почти одинакового размера: следующий класс продолжает раздачу с того блока, на котором остановился предыдущий.

In [ ]:
# Готовая ячейка: выборка с редким классом
X_rare, y_rare = [], []
n_cyclists = 0
for x, label in zip(Xr_train, y_train):
    if label == 1:
        n_cyclists += 1
        if n_cyclists > 10:
            continue                      # остальных велосипедистов пропускаем
    X_rare.append(x); y_rare.append(label)
print('объектов:', len(y_rare), ' по классам:', count_classes(y_rare), '  ожидается 151, {2: 74, 0: 67, 1: 10}')

In [ ]:
def count_in_folds(y, folds, label):
    counts = []
    for fold in folds:
        ...                                     # <-- сколько объектов класса label в блоке
    return counts

def stratified_k_fold_indices(y, n_folds, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    by_class = {}
    for i, label in enumerate(y):
        ...                                     # <-- номер i в список своего класса
    folds = [[] for _ in range(n_folds)]
    pos = 0
    for label in sorted(by_class):
        idx = by_class[label]
        rnd.shuffle(idx)
        for i in idx:
            ...                                 # <-- раздаем по кругу
            pos += 1
    return folds


print(stratified_k_fold_indices([0, 0, 0, 0, 1, 1, 1, 2, 2], 3, seed=0), '  ожидается [[2, 3, 5], [0, 4, 7], [1, 6, 8]]')

plain = k_fold_indices(len(y_rare), 5)
strat = stratified_k_fold_indices(y_rare, 5)
print('велосипедистов по блокам, обычная нарезка:        ', count_in_folds(y_rare, plain, 1), '  ожидается [0, 2, 4, 1, 3]')
print('велосипедистов по блокам, стратифицированная:     ', count_in_folds(y_rare, strat, 1), '  ожидается [2, 2, 2, 2, 2]')
print('размеры стратифицированных блоков:', [len(f) for f in strat], '  ожидается [31, 30, 30, 30, 30]')

bad = 0
for seed in range(100):
    if 0 in count_in_folds(y_rare, k_fold_indices(len(y_rare), 5, seed), 1):
        bad += 1
print('обычных нарезок, где в каком-то блоке нет ни одного велосипедиста:', bad, 'из 100   ожидается 49')

**Вопрос К4.** а) Что происходит в раунде обычной кросс-валидации, где в валидационном блоке нет ни одного велосипедиста? Какие ошибки модели мы в этом раунде не увидим? б) А если велосипедистов нет в обучающей части раунда? в) Нужна ли стратификация, если классы в выборке представлены почти поровну?

*Ваш ответ:*

**Задание К5. Сколько блоков взять? Скользящий контроль по одному объекту.**

- `loo_folds(n)` – блоки для скользящего контроля по одному объекту: $n$ блоков, в каждом один номер: `[[0], [1], …, [n-1]]`. Перемешивать не нужно: при $K = n$ порядок раундов не влияет на результат.
- В цикле по $K \in \{2, 3, 5, 10\}$ получите стратифицированные блоки и оценку kNN с $k = 9$. Затем посчитайте скользящий контроль по одному объекту: при $K = n$ точность каждого раунда равна 1 (объект угадан) или 0 (не угадан), а среднее – доля угаданных объектов.

Обратите внимание, какую долю данных видит модель при обучении: при $K$ блоках это $\frac{K-1}{K}$ выборки.

In [ ]:
def loo_folds(n):
    return ...                          # <-- n блоков по одному номеру


print(loo_folds(3), '  ожидается [[0], [1], [2]]')
print(' K  обучение   CV      σ')
for K in [2, 3, 5, 10]:
    folds = ...                                 # <-- K стратифицированных блоков
    m, s = ...                                  # <-- оценка kNN, k = 9
    print(f'{K:2d}   {210 - len(folds[0]):4d}     {m:.3f}   {s:.3f}')
print('  ожидается: K = 2: 0.848 и 0.029;  3: 0.867 и 0.027;  5: 0.886 и 0.038;  10: 0.886 и 0.074')

loo_scores = ...                                # <-- скользящий контроль
print(f'по одному объекту: {len(loo_scores)} обучений, угадано {int(sum(loo_scores))} из 210, CV = {mean_std(loo_scores)[0]:.3f}')
print('  ожидается: 210 обучений, угадано 184 из 210, CV = 0.876')

**Вопрос К5.** а) Почему при $K = 2$ оценка заметно ниже, чем при $K = 5$ и $K = 10$? Подсказка: сколько объектов видит модель при обучении? б) Почему $\sigma$ растет с ростом $K$? Значит ли это, что оценка при $K = 10$ хуже? в) Почему скользящий контроль по одному объекту дает одно и то же число при любом `seed`? Чем он плох для модели, которая обучается минуту?

*Ваш ответ:*

**Задание К6. Перебор по сетке.**

- `make_grid(options)` – все комбинации значений гиперпараметров. `options` – словарь «название → список значений», например `{'k': [1, 3], 'dist': ['euclidean', 'manhattan']}`, а ответ – список словарей:
  `[{'k': 1, 'dist': 'euclidean'}, {'k': 1, 'dist': 'manhattan'}, {'k': 3, 'dist': 'euclidean'}, {'k': 3, 'dist': 'manhattan'}]`.
  Идея: начинаем со списка из одной пустой комбинации `[{}]` и для каждого гиперпараметра дополняем **каждую** уже готовую комбинацию **каждым** его значением. Новый словарь с добавленным ключом удобно строить так: `{**p, name: v}` (этот прием был в `set_column` из Б1.6).
- `grid_search(model, X, y, folds, grid)` – для каждой комбинации `params` из сетки посчитать среднее и $\sigma$ кросс-валидации, сложить тройку `(params, среднее, σ)` в список `results` и запомнить комбинацию с наибольшим средним. Сравнение строгое (`>`): при равенстве остается комбинация, которая встретилась раньше. Возвращает `(best_params, results)`.

Все комбинации проверяем на **одних и тех же** блоках `folds` – так сравнение честное.

In [ ]:
def make_grid(options):
    grid = [{}]
    for name, values in options.items():
        grid = ...                              # <-- каждую комбинацию дополняем каждым значением
    return grid

def grid_search(model, X, y, folds, grid):
    results = []
    best_params, best_mean = None, -1
    for params in grid:
        m, s = ...                              # <-- CV для комбинации params
        results.append((params, m, s))
        if m > best_mean:
            best_params, best_mean = ...        # <-- запомнить лучшую комбинацию
    return best_params, results


print(make_grid({'k': [1, 3], 'dist': ['euclidean', 'manhattan']}))
print(len(make_grid({'a': [1, 2, 3], 'b': [1, 2], 'c': [1, 2, 3, 4]})), '  ожидается 24 = 3 · 2 · 4')

folds_r = stratified_k_fold_indices(y_train, 5)
ks = list(range(1, 40, 2))
best, results = grid_search(knn_model, Xr_train, y_train, folds_r, make_grid({'k': ks}))
for params, m, s in results:
    if params['k'] in (1, 9, 13, 39):
        print(f"k = {params['k']:2d}   CV = {m:.3f} ± {s:.3f}")
print('  ожидается: k = 1: 0.829;  k = 9: 0.886;  k = 13: 0.890;  k = 39: 0.848')
print('лучшее:', best, '  ожидается {\'k\': 13}')

best2, results2 = grid_search(knn_model, Xr_train, y_train, folds_r,
                              make_grid({'k': ks, 'dist': ['euclidean', 'manhattan']}))
print('комбинаций:', len(results2), ' обучений:', len(results2) * 5, '  ожидается 40 и 200')
print('лучшее:', best2, '  ожидается {\'k\': 13, \'dist\': \'euclidean\'}')

In [ ]:
# Готовая ячейка: кривая кросс-валидации для k
for dist, color in [('euclidean', BLUE), ('manhattan', GREY)]:
    rows = [(p['k'], m, s) for p, m, s in results2 if p['dist'] == dist]
    kk, mm, ss = [r[0] for r in rows], [r[1] for r in rows], [r[2] for r in rows]
    plt.plot(kk, mm, marker='o', color=color, label=f'{dist}: CV')
    if dist == 'euclidean':
        plt.fill_between(kk, [a - b for a, b in zip(mm, ss)], [a + b for a, b in zip(mm, ss)],
                         color=BLUE, alpha=0.15, label='euclidean: CV ± σ')
plt.axvline(best2['k'], color=NAVY, linestyle='--', label=f"лучшее k = {best2['k']}")
plt.xlabel('k – число соседей'); plt.ylabel('точность, кросс-валидация')
plt.title('Мало соседей – шум, много соседей – «средняя температура»')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

**Вопрос К6.** а) Лучшим оказалось $k = 13$ с CV 0,890, а у $k = 7$ и $k = 9$ – 0,886. Разница – один объект из 210. Можно ли сказать, что $k = 13$ точно лучше? Сравните разницу с $\sigma$. б) Сколько обучений понадобится, если добавить третий гиперпараметр с 10 значениями? А если блоков будет 10? в) Почему все комбинации нужно проверять на одних и тех же блоках?

*Ваш ответ:*

**Задание К7. Глубина дерева на Wine и итоговая проверка.**

Выполните полный протокол из п. 8 теории для дерева решений:

1. получите 5 стратифицированных блоков для обучающей выборки Wine (`yw_train`);
2. переберите `max_depth` от 1 до 8 функцией `grid_search` с моделью `tree_model`;
3. обучите дерево с лучшей глубиной на **всей** обучающей выборке (`build_tree`);
4. один раз посчитайте точность на тестовой выборке.

Дереву масштабирование не нужно, поэтому берем исходные признаки Wine.

In [ ]:
folds_w = ...                                   # <-- 5 стратифицированных блоков
grid_tree = ...                                 # <-- сетка: глубина от 1 до 8
best_tree, results_tree = ...                   # <-- перебор по сетке
for params, m, s in results_tree:
    print(f"max_depth = {params['max_depth']}   CV = {m:.3f} ± {s:.3f}")
print('  ожидается: 0.680, 0.904, затем 0.912 для всех глубин от 3 до 8')
print('лучшее:', best_tree, '  ожидается {\'max_depth\': 3}')

final_tree = ...                                # <-- обучение на всей обучающей выборке
test_acc_tree = ...                             # <-- один раз на тесте
print(f'точность на тесте: {test_acc_tree:.3f}   ожидается 0.868')

In [ ]:
# Готовая ячейка: CV для разной глубины дерева
depths = [p['max_depth'] for p, m, s in results_tree]
plt.figure(figsize=(7, 3.3))
plt.errorbar(depths, [m for p, m, s in results_tree], yerr=[s for p, m, s in results_tree],
             marker='o', color=BLUE, ecolor=GREY, capsize=4)
plt.axvline(best_tree['max_depth'], color=NAVY, linestyle='--', label=f"выбрано: max_depth = {best_tree['max_depth']}")
plt.xlabel('max_depth'); plt.ylabel('точность, CV ± σ'); plt.title('Wine: дерево решений, 5 блоков')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Вопрос К7.** а) Глубины от 3 до 8 дали одинаковую оценку 0,912. Почему `grid_search` выбрал именно 3 и почему это разумно? б) Выведите `cross_val_score` для глубин 4 и 8: точности по блокам совпадают полностью. Почему? в) Оценка кросс-валидации 0,912, а на тесте 0,868. Ошибка ли это? Можно ли теперь вернуться и выбрать глубину по тесту?

*Ваш ответ:*

**Задание К8. Подготовка данных внутри каждого блока.**

Вернемся к «грязному» набору Wine из Б1.6: примерно 5 % значений пропали, а 2 % умножены на 10 (лишний ноль при вводе). kNN с пропусками не запустится: расстояние до `None` не посчитать. В Б1.6 мы написали `prepare(X_train, X_test, remove_outliers)`: выбросы превращаются в пропуски, пропуски заполняются медианой, признаки стандартизируются, причем **все параметры считаются по первой выборке** и применяются к обеим. Функция уже готова в ячейке ниже.

В кросс-валидации роль тестовой выборки играет валидационный блок. Напишите `cross_val_score_prep(model, prep, X, y, folds, params)` – ту же кросс-валидацию, но в каждом раунде сначала подготовить данные: `X_tr, X_val = prep(X_tr, X_val)`, где параметры подготовки считаются только по обучающей части раунда. Затем обучить модель и посчитать точность.

Сравните две подготовки: полную (`prep_full`) и без обработки выбросов (`prep_no_outliers`). Модель – kNN с $k = 5$, как в Б1.6.

In [ ]:
# Готовая ячейка: подготовка данных из Б1.6
def known(values):
    return [v for v in values if v is not None]

def median(values):
    v = sorted(known(values)); n = len(v); m = n // 2
    return v[m] if n % 2 == 1 else (v[m - 1] + v[m]) / 2

def quantile(values, q):
    v = sorted(known(values)); pos = (len(v) - 1) * q; i = int(pos); f = pos - i
    return v[i] if i + 1 >= len(v) else v[i] + f * (v[i + 1] - v[i])

def iqr_bounds(values, k=1.5):
    q1, q3 = quantile(values, 0.25), quantile(values, 0.75)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def outliers_to_none(values, low, high):
    return [None if v is None or v < low or v > high else v for v in values]

def fill_missing(values, fill_value):
    return [fill_value if v is None else v for v in values]

def fit_standard(values):
    mu = sum(values) / len(values)
    return mu, math.sqrt(sum((x - mu) ** 2 for x in values) / len(values))

def apply_standard(values, params):
    mu, sigma = params
    return [(x - mu) / sigma for x in values]

def prepare(X_train, X_test, remove_outliers=True):
    """Все параметры (границы выбросов, медиана, μ, σ) – только по X_train, применяются к обеим выборкам."""
    A, B = [list(r) for r in X_train], [list(r) for r in X_test]
    for j in range(len(A[0])):
        col_a, col_b = [r[j] for r in A], [r[j] for r in B]
        if remove_outliers:
            low, high = iqr_bounds(col_a)
            col_a, col_b = outliers_to_none(col_a, low, high), outliers_to_none(col_b, low, high)
        fill = median(col_a)
        col_a, col_b = fill_missing(col_a, fill), fill_missing(col_b, fill)
        params = fit_standard(col_a)
        col_a, col_b = apply_standard(col_a, params), apply_standard(col_b, params)
        for r, v in zip(A, col_a): r[j] = v
        for r, v in zip(B, col_b): r[j] = v
    return A, B

def make_dirty(X, seed, p_missing=0.05, p_outlier=0.02):
    rnd = random.Random(seed)
    result = []
    for row in X:
        new_row = []
        for v in row:
            u = rnd.random()
            if u < p_missing:
                new_row.append(None)
            elif u < p_missing + p_outlier:
                new_row.append(v * 10)
            else:
                new_row.append(v)
        result.append(new_row)
    return result

def prep_full(X_tr, X_val):
    return prepare(X_tr, X_val, remove_outliers=True)

def prep_no_outliers(X_tr, X_val):
    return prepare(X_tr, X_val, remove_outliers=False)

D_train = make_dirty(Xw_train, seed=1)                  # тот же «грязный» Wine, что в Б1.6
print('пропусков в обучающей выборке:', sum(1 for r in D_train for v in r if v is None))

In [ ]:
def cross_val_score_prep(model, prep, X, y, folds, params):
    scores = []
    for i in range(len(folds)):
        X_tr, y_tr, X_val, y_val = split_by_fold(X, y, folds, i)
        X_tr, X_val = ...                       # <-- параметры подготовки – только по X_tr
        y_pred = ...                            # <-- обучение и прогноз
        ...                                     # <-- точность на блоке i
    return scores


full = cross_val_score_prep(knn_model, prep_full, D_train, yw_train, folds_w, {'k': 5})
no_out = cross_val_score_prep(knn_model, prep_no_outliers, D_train, yw_train, folds_w, {'k': 5})
print(f'полная подготовка:        CV = {mean_std(full)[0]:.3f} ± {mean_std(full)[1]:.3f}   ожидается 0.984 ± 0.020')
print(f'без обработки выбросов:   CV = {mean_std(no_out)[0]:.3f} ± {mean_std(no_out)[1]:.3f}   ожидается 0.816 ± 0.070')

**Вопрос К8.** а) Почему нельзя один раз вызвать `prepare(D_train, D_train)` для всей обучающей выборки, а потом запустить обычную `cross_val_score`? Какие числа при этом «подсмотрят» валидационный блок? б) Мы сравнили две подготовки кросс-валидацией и выбрали лучшую. Чем параметр `remove_outliers` отличается от гиперпараметра $k$? в) В sklearn для этого есть `Pipeline`: подготовка и модель склеиваются в одну модель. Почему это удобно именно для кросс-валидации?

*Ваш ответ:*

**Задание К9. Утечка: 80 % на случайных числах.**

Готовая функция `make_noise` создает 60 объектов со 1000 признаков из **случайных** чисел и **случайными** ответами 0 и 1 (по 30 каждого). Никакой связи между признаками и ответом нет, поэтому честная модель угадывает примерно в 50 % случаев.

Когда признаков много, их часто отбирают: оставляют те, что сильнее всего отличаются у двух классов. Отбор признаков – тоже обучение по данным, поэтому пишем его в виде пары fit/apply, как в Б1.6:

- `fit_selector(X, y, n_keep)` – для каждого признака $j$ посчитать модуль разности средних у класса 1 и класса 0: $\left|\bar x_j^{(1)} - \bar x_j^{(0)}\right|$, и вернуть номера `n_keep` признаков с наибольшей разностью. Сортировка номеров по значениям списка: `sorted(range(len(scores)), key=lambda j: scores[j], reverse=True)`.
- `apply_selector(X, cols)` – оставить в каждой строке только столбцы с номерами из `cols`.

Эксперимент в два шага:

1. **Неправильно:** отобрать 10 признаков по **всей** таблице, потом запустить кросс-валидацию kNN ($k = 5$).
2. **Правильно:** в каждом раунде отбирать признаки только по обучающей части раунда, применять отбор к обеим частям, затем обучать kNN.

In [ ]:
# Готовая ячейка: случайные данные без всякой закономерности
def make_noise(n=60, n_features=1000, seed=0):
    rnd = random.Random(seed)
    X = [[rnd.random() for _ in range(n_features)] for _ in range(n)]
    y = [i % 2 for i in range(n)]
    rnd.shuffle(y)
    return X, y

X_noise, y_noise = make_noise()
folds_n = stratified_k_fold_indices(y_noise, 5)
print(len(X_noise), 'объектов,', len(X_noise[0]), 'признаков, классы:', count_classes(y_noise))

In [ ]:
def fit_selector(X, y, n_keep):
    scores = []
    for j in range(len(X[0])):
        ones = [row[j] for row, label in zip(X, y) if label == 1]
        zeros = [row[j] for row, label in zip(X, y) if label == 0]
        ...                                     # <-- модуль разности средних
    order = sorted(range(len(scores)), key=lambda j: scores[j], reverse=True)
    return ...                                  # <-- номера n_keep лучших признаков

def apply_selector(X, cols):
    return ...                                  # <-- только столбцы из cols


print(apply_selector([[10, 11, 12], [20, 21, 22]], [2, 0]), '  ожидается [[12, 10], [22, 20]]')
print(fit_selector([[1, 5], [2, 5], [9, 6], [8, 6]], [0, 0, 1, 1], 1), '  ожидается [0]')

# 1) Неправильно: отбор по всей таблице, потом кросс-валидация
cols_all = ...                                  # <-- отбор по всей таблице
leaky = cross_val_score(knn_model, apply_selector(X_noise, cols_all), y_noise, folds_n, {'k': 5})

# 2) Правильно: отбор внутри каждого раунда
honest = []
for i in range(len(folds_n)):
    X_tr, y_tr, X_val, y_val = split_by_fold(X_noise, y_noise, folds_n, i)
    cols = ...                                  # <-- отбор только по обучающей части раунда
    y_pred = ...                                # <-- обучение и прогноз
    honest.append(accuracy(y_val, y_pred))

print(f'отбор по всей таблице:  CV = {mean_std(leaky)[0]:.3f}   ожидается 0.817')
print(f'отбор внутри раундов:   CV = {mean_std(honest)[0]:.3f}   ожидается 0.433')

**Вопрос К9.** а) Откуда 82 % точности на случайных числах, если закономерности нет? Что именно «подсмотрел» отбор признаков по всей таблице? б) Почему честная оценка 0,433 даже ниже 0,5? Опасно ли это? в) Представьте: вы отобрали признаки по всей таблице, получили 82 % и доложили результат. Что произойдет, когда модель встретит новые данные?

*Ваш ответ:*

**Задание К10. Сто случайных разбиений: одна валидация против кросс-валидации.**

В К1 мы получили 100 оценок одной валидационной выборки (`hold_scores`). Теперь при тех же `seed` от 0 до 99 посчитайте оценку 5-блочной кросс-валидации: при каждом `seed` новые блоки `k_fold_indices(210, 5, seed)` и среднее по 5 раундам. Сложите 100 чисел в список `cv_scores` и сравните разброс двух способов.

In [ ]:
cv_scores = []
for seed in range(100):
    folds = ...                                 # <-- новая нарезка при данном seed
    ...                                         # <-- среднее по 5 блокам

for name, values in [('одна валидация  ', hold_scores), ('кросс-валидация', cv_scores)]:
    m, s = mean_std(values)
    print(f'{name}: от {min(values):.3f} до {max(values):.3f}, среднее {m:.3f}, σ = {s:.3f}')
print('  ожидается: одна валидация от 0.794 до 0.952, σ = 0.036;  кросс-валидация от 0.848 до 0.890, σ = 0.008')
print(f'разброс меньше в {mean_std(hold_scores)[1] / mean_std(cv_scores)[1]:.1f} раза   ожидается 4.4')

In [ ]:
# Готовая ячейка: две гистограммы на одной оси
bins = [(j - 0.5) / 63 for j in range(49, 62)]
plt.figure(figsize=(7, 3.3))
plt.hist(hold_scores, bins=bins, color=GREY, alpha=0.8, edgecolor='white', label='одна валидация (63 объекта)')
plt.hist(cv_scores, bins=bins, color=BLUE, alpha=0.85, edgecolor='white', label='кросс-валидация, 5 блоков')
plt.xlabel('оценка точности kNN, k = 9'); plt.ylabel('сколько разбиений')
plt.title('Сто разбиений: насколько «прыгает» оценка')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Вопрос К10.** а) Средние у двух способов почти одинаковые (0,869 и 0,872), а разброс отличается в 4,4 раза. Что это значит для выбора $k$? б) За уменьшение разброса мы заплатили: во сколько раз больше обучений требует кросс-валидация? в) Почему у кросс-валидации разброс не нулевой, хотя каждый объект побывал в валидации?

*Ваш ответ:*

**Задание К11. Полный протокол для kNN и сверка со sklearn.**

В К6 кросс-валидация выбрала $k = 13$ (евклидово расстояние). Завершите протокол из п. 8 теории: обучите kNN с лучшими гиперпараметрами `best2` на **всей** обучающей выборке `Xr_train, y_train` (для kNN это значит: запомнить все 210 объектов), сделайте прогноз для `Xr_test` и один раз посчитайте точность на тесте.

После этого готовые ячейки покажут, что было бы, если бы мы подбирали $k$ по тесту, и сверят наш перебор с `GridSearchCV` из sklearn на тех же блоках.

In [ ]:
final_pred = ...                                # <-- обучение на всех 210 объектах, прогноз для теста
test_acc = ...                                  # <-- точность на тесте, один раз
print(f'лучшие гиперпараметры: {best2},  CV = {max(m for p, m, s in results2):.3f}')
print(f'точность на тесте: {test_acc:.3f}   ожидается 0.911')

In [ ]:
# Готовая ячейка: так делать нельзя – «подбор k по тесту»
test_by_k = [(k, accuracy(y_test, knn_model(Xr_train, y_train, Xr_test, {'k': k}))) for k in ks]
k_peek, acc_peek = max(test_by_k, key=lambda p: p[1])
cv_peek = [m for p, m, s in results if p['k'] == k_peek][0]
print(f'если выбирать по тесту: k = {k_peek}, «точность» {acc_peek:.3f}, но CV для этого k всего {cv_peek:.3f}')
print('  ожидается: k = 23, 0.944 и 0.867')

In [ ]:
# Готовая ячейка: сверка с GridSearchCV из sklearn на тех же пяти блоках
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

splits = [([j for f in range(5) if f != i for j in folds_r[f]], folds_r[i]) for i in range(5)]
gs = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': ks}, cv=splits, scoring='accuracy')
gs.fit(Xr_train, y_train)
print('sklearn:', gs.best_params_, f'CV = {gs.best_score_:.3f}   ожидается n_neighbors = 13, CV = 0.886')
print('наш:    ', best, f"CV = {max(m for p, m, s in results):.3f}")
diff = max(abs(m - sk) for (p, m, s), sk in zip(results, gs.cv_results_['mean_test_score']))
print(f'наибольшее расхождение оценок: {diff:.3f}   ожидается 0.019')
print('расхождения – это ничьи в голосовании: sklearn отдает их классу с меньшим номером, а мы – ближнему соседу')

**Вопрос К11.** а) В Б1.2 по одной валидации выбрали $k = 9$, и на тесте было 0,933. Кросс-валидация выбрала $k = 13$, и на тесте 0,911. Значит ли это, что кросс-валидация ошиблась? Сколько объектов теста стоит за разницей? б) По тесту лучшим было бы $k = 23$ с 0,944. Почему этому числу нельзя верить? в) Кросс-валидация дала для $k = 13$ оценку 0,890. Почему и эта оценка немного завышена?

*Ваш ответ:*

---
### Задания повышенной сложности

1. **Вложенная кросс-валидация.** Напишите `nested_cv(model, X, y, grid, n_outer=5, n_inner=5)`. Внешний цикл: 5 стратифицированных блоков, как обычно. В каждом внешнем раунде на обучающей части **заново** выполните `grid_search` со своей внутренней 5-блочной нарезкой, обучите модель с найденными гиперпараметрами на всей обучающей части внешнего раунда и проверьте на внешнем валидационном блоке. Для kNN на `Xr_train` и сетки $k = 1, 3, \dots, 39$ должно получиться около $0{,}857 \pm 0{,}040$, а в разных внешних раундах выбираются разные $k$ (7, 15, 5, 5, 17). Почему эта оценка ниже, чем 0,890 из К6? Что она оценивает: одну модель или всю процедуру выбора модели?
2. **Повторная кросс-валидация.** Повторите 5-блочную кросс-валидацию 10 раз с разными `seed` и усредните 50 оценок. Сравните кривую CV для $k$ с кривой из К6: стало ли плато глаже? Изменилось ли лучшее $k$?
3. **Правило одной стандартной ошибки.** Стандартная ошибка среднего по $K$ блокам равна $\sigma / \sqrt{K}$. Правило: среди всех комбинаций, у которых CV не ниже $\mathrm{CV}_{\max} - \sigma_{\max} / \sqrt{K}$, выбрать **самую простую** модель (для kNN – с наибольшим $k$, для дерева – с наименьшей глубиной). Реализуйте правило для результатов К6 и К7. Ответ: $k = 21$ и глубина 2.
4. **Случайный поиск.** Вместо полной сетки $k \in \{1, 3, \dots, 99\}$ × 2 расстояния (100 комбинаций) проверьте 15 случайных комбинаций (`random.Random(seed).sample`). Насколько лучший результат случайного поиска хуже полного перебора и во сколько раз меньше обучений? Повторите для 10 разных `seed`.
5. **Кадры одного видео.** Беспилотник видит один и тот же объект на нескольких соседних кадрах, и признаки на них почти совпадают. Сгенерируйте данные, где каждый объект из `X_road` повторяется в 5 кадрах с небольшим шумом (`rnd.gauss(0, 0.02)` к нормализованным признакам), и сравните обычную кросс-валидацию с групповой: `group_k_fold_indices(groups, n_folds)` кладет все кадры одного объекта в один блок. Почему обычная нарезка здесь дает завышенную оценку, особенно для $k = 1$?

In [ ]:
# Место для заданий повышенной сложности